# Preparacion de Datos - Speed Dating Columbia University

Este notebook realiza la preparacion completa de los datos para el analisis de citas rapidas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

PALETA_ROSA = ["#FF1493", "#FF69B4", "#FFB6C1", "#FFC0CB"]
sns.set_palette(PALETA_ROSA)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

print("Librerias importadas correctamente")

## 1. Carga de Datos

In [ ]:
data_path = "../data/Speed Dating Data.csv"
df = pd.read_csv(data_path, encoding="latin-1")

print(f"Dimensiones: {df.shape}")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()

## 2. Diccionario de Variables

In [ ]:
diccionario_variables = {
    "iid": "Identificador del participante",
    "id": "Identificador de la cita",
    "gender": "Genero",
    "age": "Edad",
    "match": "Match - 1=si 0=no"
}

df_dict = pd.DataFrame(list(diccionario_variables.items()), columns=["Variable", "Descripcion"])
print(f"Variables ({len(df_dict)}):")
df_dict

## 3. Pandas Profiling

In [ ]:
import os
os.makedirs("../reports", exist_ok=True)

try:
    from ydata_profiling import ProfileReport
except ImportError:
    try:
        from pandas_profiling import ProfileReport
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "ydata-profiling", "-q"])
        from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Speed Dating - Analisis Exploratorio", explorative=True)
profile.to_file("../reports/pandas_profiling.html")
print("Guardado: reports/pandas_profiling.html")

## 4. Distribucion del Target

In [ ]:
match_counts = df["match"].value_counts()
print("Distribucion:")
print(match_counts)
print("Match 1: {:.1f}%".format(match_counts[1]/len(df)*100))
print("Match 0: {:.1f}%".format(match_counts[0]/len(df)*100))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

bars = axes[0].bar(["No Match", "Match"], [match_counts[0], match_counts[1]], color=["#FFB6C1", "#FF1493"], edgecolor="black")
axes[0].set_title("Distribucion", fontweight="bold")
axes[0].set_ylabel("Frecuencia")

for bar in bars:
    h = bar.get_height()
    t = "{:,.0f}".format(h)
    axes[0].text(bar.get_x() + bar.get_width()/2., h, t, ha="center", va="bottom", fontweight="bold")

axes[1].pie([match_counts[0], match_counts[1]], labels=["No Match", "Match"], autopct="%1.1f%%", colors=["#FFB6C1", "#FF1493"])
axes[1].set_title("Proporcion", fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/distribucion_target.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Estadistica Descriptiva

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Columnas numericas: {}".format(len(numeric_cols)))

descriptivas = df[numeric_cols].describe().T
descriptivas["missing_count"] = df[numeric_cols].isnull().sum()
descriptivas["missing_pct"] = (df[numeric_cols].isnull().sum() / len(df) * 100).round(2)

print("\nEstadisticas:")
descriptivas.head(10)

descriptivas.to_csv("../reports/estadisticas_descriptivas.csv")

plt.figure(figsize=(16, 14))
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap="RdBu_r", center=0, square=True, linewidths=0.5, cbar_kws={"shrink": .8}, fmt=".2f", annot=False, vmin=-1, vmax=1)
plt.title("Matriz de Correlacion", fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("../reports/matriz_correlacion_inicial.png", dpi=300, bbox_inches="tight")
plt.show()
print("Guardado en reports/")

## 6. Limpieza de Nulos

In [ ]:
df_clean = df.copy()

cols_to_drop = ["iid","id","idg","partner","pid","wave","round","position","positin1","order","condtn","undergrd","zipcode","career","from","field"]

print("Columnas a eliminar: {}".format(len(cols_to_drop)))

df_clean.drop(columns=cols_to_drop, inplace=True, errors="ignore")

print("Despues de eliminar: {} columnas".format(df_clean.shape[1]))

nulos_por_columna = df_clean.isnull().sum()
pct_nulos = (nulos_por_columna / len(df_clean) * 100).round(2)

total_nulos = df_clean.isnull().sum().sum()
print("\nValores nulos totales: {}".format(total_nulos))

columnas_mas_50 = nulos_por_columna[pct_nulos > 50].index.tolist()
print("Columnas con >50% nulos: {}".format(len(columnas_mas_50)))

if columnas_mas_50:
    df_clean.drop(columns=columnas_mas_50, inplace=True)
    print("Eliminadas: {}".format(columnas_mas_50))

df_nulos = pd.DataFrame({"Columna": nulos_por_columna.index, "Nulos": nulos_por_columna.values, "Porcentaje": pct_nulos.values})
df_nulos.to_csv("../reports/nulos_analisis.csv", index=False)
print("Analisis guardado")

## 7. Imputacion con Mediana

In [ ]:
numeric_cols_clean = df_clean.select_dtypes(include=[np.number]).columns.tolist()
if "match" in numeric_cols_clean:
    numeric_cols_clean.remove("match")

medianas = {}
for col in numeric_cols_clean:
    if col in df_clean.columns and df_clean[col].isnull().sum() > 0:
        med = df_clean[col].median()
        medianas[col] = med
        df_clean[col].fillna(med, inplace=True)
        print("{}: mediana={:.2f}".format(col, med))

total_nulos = df_clean.isnull().sum().sum()
print("\nNulos restantes: {}".format(total_nulos))

pd.DataFrame(list(medianas.items()), columns=["Columna","Mediana"]).to_csv("../reports/medianas_imputadas.csv", index=False)
print("Medianas guardadas")

## 8. Winsorizing (5-95)

In [ ]:
score_cols = [c for c in df_clean.select_dtypes(include=[np.number]).columns if c not in ["match","gender","race","race_o","samerace","goal","date","go_out","field_cd","career_c","met","dec","dec_o","int_corr"]]

print("Winsorizando {} columnas...".format(len(score_cols)))

for col in score_cols:
    if col in df_clean.columns:
        p5 = df_clean[col].quantile(0.05)
        p95 = df_clean[col].quantile(0.95)
        df_clean[col] = df_clean[col].clip(lower=p5, upper=p95)

print("Winsorizing completado")
print("Shape final: {}".format(df_clean.shape))

df_clean.to_csv("../data/data_prepared.csv", index=False)
print("Datos preparados guardados")

## 9. Reduccion de Redundancia

In [ ]:
print("Correlacion con target:")
corr_m = df_clean.corr()
target_c = corr_m["match"].abs().sort_values(ascending=False)
print(target_c.head(20))

irrelevantes = target_c[target_c < 0.02].index.tolist()
irrelevantes = [c for c in irrelevantes if c != "match"]
print("\nIrrelevantes (<0.02 con match): {}".format(len(irrelevantes)))

redundantes = []
corr_abs = corr_m.abs()
np.fill_diagonal(corr_abs.values, 0)

for i in range(len(corr_abs.columns)):
    for j in range(i+1, len(corr_abs.columns)):
        if corr_abs.iloc[i,j] > 0.85:
            redundantes.append((corr_abs.columns[i], corr_abs.columns[j], corr_abs.iloc[i,j]))

print("Pares redundantes (corr > 0.85): {}".format(len(redundantes)))
for r in redundantes[:10]:
    print("  {} - {}: {:.3f}".format(r[0], r[1], r[2]))